In [3]:
import pandas as pd
import numpy as np
import re

# 0. 자치구 코드 매핑
# DATA/TARGET/CODE1.csv 기반
gu_code_map = {
    "종로구": 11110, "중구": 11140, "용산구": 11170, "성동구": 11200,
    "광진구": 11215, "동대문구": 11230, "중랑구": 11260, "성북구": 11290,
    "강북구": 11305, "도봉구": 11320, "노원구": 11350, "은평구": 11380,
    "서대문구": 11410, "마포구": 11440, "양천구": 11470, "강서구": 11500,
    "구로구": 11530, "금천구": 11545, "영등포구": 11560, "동작구": 11590,
    "관악구": 11620, "서초구": 11650, "강남구": 11680, "송파구": 11710,
    "강동구": 11740,
}

# 1) S-DoT 유동인구 측정 정보
num = pd.read_parquet(
    "../DATA/TARGET/NUM/merged_data.parquet"
)

# 2) S-DoT 센서 설치 위치 정보
space1 = pd.read_csv(
    "../DATA/TARGET/SPACE1.csv"
)

# 3) 보도 세부 현황
space2 = pd.read_csv(
    "../DATA/TARGET/SPACE2.csv"
)

# 2. 컬럼명 공백 정리

num.columns = num.columns.str.strip()
space1.columns = space1.columns.str.strip()
space2.columns = space2.columns.str.strip()

print("num columns:", num.columns.tolist())
print("space1 columns:", space1.columns.tolist())
print("space2 columns:", space2.columns.tolist())

num columns: ['모델번호', '시리얼', '측정시간', '지역', '자치구', '행정동', '방문자수', '등록일']
space1 columns: ['순번', '방문자 센서코드', '시리얼번호', '주소', '위도', '경도']
space2 columns: ['연번', '노선번호', '관리기관', '노선명', '방향', '구간(위치)', '연장(m)', '폭(m)', '면적(m2)', '블록 종류', '투수', '시공년도', '횡단구성(6)', '비 고(7)', '총연장', '1707728.8', '총면적', '6,808,873.4']


In [7]:
# 3. space2 정리

space2_clean = space2.copy()

# 자치구코드 생성
space2_clean["자치구코드"] = space2_clean["관리기관"].map(gu_code_map)

# 면적 숫자 변환
space2_clean["면적_m2"] = pd.to_numeric(
    space2_clean["면적(m2)"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce"
)

# 연장 숫자 변환
space2_clean["연장_m"] = pd.to_numeric(
    space2_clean["연장(m)"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce"
)

# 유효 행만 남기기
space2_clean = space2_clean.dropna(subset=["자치구코드", "면적_m2"])
space2_clean = space2_clean[space2_clean["면적_m2"] > 0].copy()

space2_clean["자치구코드"] = space2_clean["자치구코드"].astype(int)

space2_clean.head()

,연번,노선번호,관리기관,노선명,방향,구간(위치),연장(m),폭(m),면적(m2),블록 종류,...,시공년도,횡단구성(6),비 고(7),총연장,1707728.8,총면적,"6,808,873.4",자치구코드,면적_m2,연장_m
0,1.0,1.0,강북구,4.19로,국립4.19묘지입구사거리 방향,4.19로 31~4.19로 1,310.0,2.5,775.0,콘크리트가공블록,...,2021.0,보도,NaN,NaN,NaN,NaN,NaN,11305,775.0,310.0
1,2.0,1.0,강북구,4.19로,국립4.19묘지교차로 방향,4.19로 2~4.19로 36-1,335.0,2.8,921.3,콘크리트가공블록,...,2021.0,보도,NaN,NaN,NaN,NaN,NaN,11305,921.3,335.0
2,3.0,2.0,강서구,가로공원로,부천시계 방향,화곡터널입구~월정초교앞,782.0,4.0,"3,128.0",콘크리트가공블록,...,2013.0,보도,NaN,NaN,NaN,NaN,NaN,11500,3128.0,782.0
3,4.0,2.0,양천구,가로공원로,부천시계 방향,월정초교앞~남부순환로,501.0,4.0,"2,004.0",소형고압블록(U형 S형 I2형 O형 등),...,2006.0,보도,NaN,NaN,NaN,NaN,NaN,11470,2004.0,501.0
4,5.0,2.0,양천구,가로공원로,부천시계 방향,남부순환로~부천시계,515.0,4.0,"2,060.0",소형고압블록(U형 S형 I2형 O형 등),...,2006.0,보도,NaN,NaN,NaN,NaN,NaN,11470,2060.0,515.0


In [8]:
# 4. space1 정리

space1_clean = space1.copy()
space1_clean.head()

,순번,방문자 센서코드,시리얼번호,주소,위도,경도
0,1,2992,V02Q1940942,서울특별시 양천구 신월동 805,37.532463,126.833076
1,2,2993,V02Q1940889,서울특별시 도봉구 창동 585-13,37.639946,127.036011
2,3,2994,V02Q1940947,서울특별시 동작구 상도동 518-7,37.499026,126.952012
3,4,2995,V02Q1940888,서울특별시 광진구 중곡동 265-5,37.563074,127.081995
4,5,2996,V02Q1940851,서울특별시 강동구 암사동 501-4,37.550391,127.128292


In [9]:
# 5. num 정리

num_clean = num.copy()

num_clean.head()
num_clean.columns

Index(['모델번호', '시리얼', '측정시간', '지역', '자치구', '행정동', '방문자수', '등록일'], dtype='str')

In [10]:
print("num shape:", num_clean.shape)
print("space1 shape:", space1_clean.shape)
print("space2 shape:", space2_clean.shape)

print(space2_clean[["관리기관", "자치구코드", "면적_m2"]].head())

num shape: (4834088, 8)
space1 shape: (108, 6)
space2 shape: (5421, 21)
  관리기관  자치구코드   면적_m2
0  강북구  11305   775.0
1  강북구  11305   921.3
2  강서구  11500  3128.0
3  양천구  11470  2004.0
4  양천구  11470  2060.0


In [11]:
print("num columns")
print(num_clean.columns.tolist())

print("\nspace1 columns")
print(space1_clean.columns.tolist())

num columns
['모델번호', '시리얼', '측정시간', '지역', '자치구', '행정동', '방문자수', '등록일']

space1 columns
['순번', '방문자 센서코드', '시리얼번호', '주소', '위도', '경도']


In [12]:
display(num_clean.head())
display(space1_clean.head())

,모델번호,시리얼,측정시간,지역,자치구,행정동,방문자수,등록일
0,SDOT001,4065,2025-06-08_23:51:00,parks,Seoul_Grand_Park,trail,0,2025-06-09 00:08:05
1,SDOT001,4015,2025-06-08_23:50:00,main_street,Jung-gu,Gwanghui-dong,138,2025-06-09 00:08:05
2,SDOT001,4036,2025-06-08_23:50:00,main_street,Yangcheon-gu,Sinjeong4(sa)-dong,11,2025-06-09 00:08:05
3,SDOT001,4048,2025-06-08_23:54:00,main_street,Gwangjin-gu,Guui1(il)-dong,0,2025-06-09 00:08:06
4,SDOT001,3037,2025-06-08_23:56:00,public_facilities,Geumcheon-gu,Doksan4(sa)-dong,113,2025-06-09 00:08:06


,순번,방문자 센서코드,시리얼번호,주소,위도,경도
0,1,2992,V02Q1940942,서울특별시 양천구 신월동 805,37.532463,126.833076
1,2,2993,V02Q1940889,서울특별시 도봉구 창동 585-13,37.639946,127.036011
2,3,2994,V02Q1940947,서울특별시 동작구 상도동 518-7,37.499026,126.952012
3,4,2995,V02Q1940888,서울특별시 광진구 중곡동 265-5,37.563074,127.081995
4,5,2996,V02Q1940851,서울특별시 강동구 암사동 501-4,37.550391,127.128292


In [13]:
# 타입 통일
num_clean["시리얼"] = num_clean["시리얼"].astype(int)
space1_clean["방문자 센서코드"] = space1_clean["방문자 센서코드"].astype(int)

# 센서 위치 붙이기
num_sensor = num_clean.merge(
    space1_clean[["방문자 센서코드", "주소", "위도", "경도"]],
    left_on="시리얼",
    right_on="방문자 센서코드",
    how="left"
)

num_sensor.head()

,모델번호,시리얼,측정시간,지역,자치구,행정동,방문자수,등록일,방문자 센서코드,주소,위도,경도
0,SDOT001,4065,2025-06-08_23:51:00,parks,Seoul_Grand_Park,trail,0,2025-06-09 00:08:05,4065.0,서울대공원 탐방로,37.427886,127.011426
1,SDOT001,4015,2025-06-08_23:50:00,main_street,Jung-gu,Gwanghui-dong,138,2025-06-09 00:08:05,4015.0,서울특별시 중구 을지로6가 18-185,37.567870,127.008631
2,SDOT001,4036,2025-06-08_23:50:00,main_street,Yangcheon-gu,Sinjeong4(sa)-dong,11,2025-06-09 00:08:05,4036.0,서울특별시 양천구 신정동 994-1,37.525918,126.863558
3,SDOT001,4048,2025-06-08_23:54:00,main_street,Gwangjin-gu,Guui1(il)-dong,0,2025-06-09 00:08:06,4048.0,서울특별시 광진구 구의동 246-8,37.537680,127.085212
4,SDOT001,3037,2025-06-08_23:56:00,public_facilities,Geumcheon-gu,Doksan4(sa)-dong,113,2025-06-09 00:08:06,3037.0,서울특별시 금천구 독산동 독산로54길 114,37.467564,126.908102


In [14]:
print("전체 행 수:", len(num_sensor))
print("위도 결측:", num_sensor["위도"].isna().sum())
print("경도 결측:", num_sensor["경도"].isna().sum())

전체 행 수: 4834088
위도 결측: 285158
경도 결측: 285158


In [15]:
num_sensor["datetime"] = pd.to_datetime(
    num_sensor["측정시간"].str.replace("_", " "),
    errors="coerce"
)

num_sensor["date"] = num_sensor["datetime"].dt.date
num_sensor["hour"] = num_sensor["datetime"].dt.hour
num_sensor["weekday"] = num_sensor["datetime"].dt.weekday
num_sensor["month"] = num_sensor["datetime"].dt.month

In [16]:
gu_eng_to_kor = {
    "Jongno-gu": "종로구",
    "Jung-gu": "중구",
    "Yongsan-gu": "용산구",
    "Seongdong-gu": "성동구",
    "Gwangjin-gu": "광진구",
    "Dongdaemun-gu": "동대문구",
    "Jungnang-gu": "중랑구",
    "Seongbuk-gu": "성북구",
    "Gangbuk-gu": "강북구",
    "Dobong-gu": "도봉구",
    "Nowon-gu": "노원구",
    "Eunpyeong-gu": "은평구",
    "Seodaemun-gu": "서대문구",
    "Mapo-gu": "마포구",
    "Yangcheon-gu": "양천구",
    "Gangseo-gu": "강서구",
    "Guro-gu": "구로구",
    "Geumcheon-gu": "금천구",
    "Yeongdeungpo-gu": "영등포구",
    "Dongjak-gu": "동작구",
    "Gwanak-gu": "관악구",
    "Seocho-gu": "서초구",
    "Gangnam-gu": "강남구",
    "Songpa-gu": "송파구",
    "Gangdong-gu": "강동구",
}

num_sensor["자치구명"] = num_sensor["자치구"].map(gu_eng_to_kor)
num_sensor["자치구코드"] = num_sensor["자치구명"].map(gu_code_map)

In [17]:
num_sensor[["시리얼", "주소", "위도", "경도", "자치구", "자치구명", "자치구코드", "방문자수", "datetime"]].head()

,시리얼,주소,위도,경도,자치구,자치구명,자치구코드,방문자수,datetime
0,4065,서울대공원 탐방로,37.427886,127.011426,Seoul_Grand_Park,NaN,NaN,0,2025-06-08 23:51:00
1,4015,서울특별시 중구 을지로6가 18-185,37.567870,127.008631,Jung-gu,중구,11140.0,138,2025-06-08 23:50:00
2,4036,서울특별시 양천구 신정동 994-1,37.525918,126.863558,Yangcheon-gu,양천구,11470.0,11,2025-06-08 23:50:00
3,4048,서울특별시 광진구 구의동 246-8,37.537680,127.085212,Gwangjin-gu,광진구,11215.0,0,2025-06-08 23:54:00
4,3037,서울특별시 금천구 독산동 독산로54길 114,37.467564,126.908102,Geumcheon-gu,금천구,11545.0,113,2025-06-08 23:56:00


In [18]:
print("자치구명 결측:", num_sensor["자치구명"].isna().sum())
print("자치구코드 결측:", num_sensor["자치구코드"].isna().sum())
print(num_sensor["자치구"].unique())

자치구명 결측: 367397
자치구코드 결측: 367397
<ArrowStringArray>
['Seoul_Grand_Park',          'Jung-gu',     'Yangcheon-gu',
      'Gwangjin-gu',     'Geumcheon-gu',        'Dobong-gu',
      'Gangdong-gu',         'Nowon-gu',       'Gangseo-gu',
       'Gangnam-gu',     'Seodaemun-gu',        'Seocho-gu',
       'Dongjak-gu',       'Gangbuk-gu',        'Gwanak-gu',
          'Guro-gu',      'Jungnang-gu',        'Jongno-gu',
        'Songpa-gu',          'Mapo-gu',       'Yongsan-gu',
     'Eunpyeong-gu',     'Seongdong-gu',  'Yeongdeungpo-gu']
Length: 24, dtype: str


In [19]:
# 위치정보가 안 붙은 시리얼 확인
missing_serials = num_sensor.loc[num_sensor["위도"].isna(), "시리얼"].unique()

print("위치정보 없는 시리얼 개수:", len(missing_serials))
print(missing_serials[:50])

위치정보 없는 시리얼 개수: 26
[4090 4075 4089 4091 4073 4078 4072 4087 4081 4082 4085 4084 4070 4094
 4074 4095 4080 4083 4088 4077 4079 4092 4093 4086 4071 4076]


In [20]:
print("num 시리얼 개수:", num_clean["시리얼"].nunique())
print("space1 센서코드 개수:", space1_clean["방문자 센서코드"].nunique())

print("num에만 있는 시리얼:", sorted(set(num_clean["시리얼"]) - set(space1_clean["방문자 센서코드"]))[:50])
print("space1에만 있는 센서코드:", sorted(set(space1_clean["방문자 센서코드"]) - set(num_clean["시리얼"]))[:50])

num 시리얼 개수: 131
space1 센서코드 개수: 108
num에만 있는 시리얼: [4070, 4071, 4072, 4073, 4074, 4075, 4076, 4077, 4078, 4079, 4080, 4081, 4082, 4083, 4084, 4085, 4086, 4087, 4088, 4089, 4090, 4091, 4092, 4093, 4094, 4095]
space1에만 있는 센서코드: [4003, 4011, 4012]


In [21]:
num_sensor.loc[num_sensor["위도"].isna(), "시리얼"].value_counts().head(20)

시리얼
4089    14180
4090    14178
4088    14168
4091    14149
4072    12826
4080    12202
4079    12177
4093    12158
4094    12108
4070    12030
4087    12017
4095    11963
4083    11929
4078    11859
4084    11808
4085    11792
4082    11713
4074    11608
4077    11489
4081    11086
Name: count, dtype: int64

In [22]:
# 위치정보 있는 센서만 사용
pop_sensor_valid = num_sensor.dropna(subset=["위도", "경도"]).copy()

print("사용 가능 행 수:", len(pop_sensor_valid))
print("제거된 행 수:", len(num_sensor) - len(pop_sensor_valid))
print("사용 가능 비율:", len(pop_sensor_valid) / len(num_sensor))

사용 가능 행 수: 4548930
제거된 행 수: 285158
사용 가능 비율: 0.9410110035233119


In [23]:
expected_gu = set(gu_code_map.keys())
actual_gu = set(num_sensor["자치구명"].unique())

print("S-DoT 데이터에 없는 자치구:", sorted(expected_gu - actual_gu))

S-DoT 데이터에 없는 자치구: ['동대문구', '성북구']


In [24]:
space1_num = (
    pop_sensor_valid
    .groupby(["시리얼", "datetime", "자치구코드", "자치구명", "위도", "경도"], as_index=False)
    .agg(
        visitor_count=("방문자수", "sum")
    )
)

space1_num.head()

,시리얼,datetime,자치구코드,자치구명,위도,경도,visitor_count
0,2992,2024-12-29 23:54:00,11470.0,양천구,37.532463,126.833076,203
1,2992,2024-12-30 00:01:00,11470.0,양천구,37.532463,126.833076,177
2,2992,2024-12-30 00:11:00,11470.0,양천구,37.532463,126.833076,193
3,2992,2024-12-30 00:21:00,11470.0,양천구,37.532463,126.833076,177
4,2992,2024-12-30 00:31:00,11470.0,양천구,37.532463,126.833076,195


In [25]:
space1_num["hour"] = space1_num["datetime"].dt.hour
space1_num["weekday"] = space1_num["datetime"].dt.weekday
space1_num["month"] = space1_num["datetime"].dt.month

space1_num["hour_sin"] = np.sin(2 * np.pi * space1_num["hour"] / 24)
space1_num["hour_cos"] = np.cos(2 * np.pi * space1_num  ["hour"] / 24)

space1_num["weekday_sin"] = np.sin(2 * np.pi * space1_num["weekday"] / 7)
space1_num["weekday_cos"] = np.cos(2 * np.pi * space1_num["weekday"] / 7)

space1_num["month_sin"] = np.sin(2 * np.pi * space1_num["month"] / 12)
space1_num["month_cos"] = np.cos(2 * np.pi * space1_num["month"] / 12)

In [26]:
gu_walk_area = (
    space2_clean
    .groupby("자치구코드")["면적_m2"]
    .sum()
    .reset_index()
    .rename(columns={"면적_m2": "gu_walk_area"})
)

sensor_count_by_gu = (
    space1_num[["시리얼", "자치구코드"]]
    .drop_duplicates()
    .groupby("자치구코드")
    .size()
    .reset_index(name="sensor_count")
)

sensor_area = gu_walk_area.merge(sensor_count_by_gu, on="자치구코드", how="left")

sensor_area["sensor_walk_area"] = sensor_area["gu_walk_area"] / sensor_area["sensor_count"]

sensor_area.head()

,자치구코드,gu_walk_area,sensor_count,sensor_walk_area
0,11110,282318.0,6.0,47053.000000
1,11140,285280.0,6.0,47546.666667
2,11170,197894.5,1.0,197894.500000
3,11200,230633.9,2.0,115316.950000
4,11215,213179.6,5.0,42635.920000


In [27]:
sensor_dataset = space1_num.merge(
    sensor_area[["자치구코드", "gu_walk_area", "sensor_count", "sensor_walk_area"]],
    on="자치구코드",
    how="left"
)

a_star = 1.39

sensor_dataset["m2_per_person"] = (
    sensor_dataset["sensor_walk_area"] / sensor_dataset["visitor_count"]
)

# 방문자수가 0이면 혼잡도 0으로 처리
sensor_dataset.loc[sensor_dataset["visitor_count"] <= 0, "m2_per_person"] = np.inf

sensor_dataset["OTI"] = 1 - (sensor_dataset["m2_per_person"] / a_star)
sensor_dataset["OTI"] = sensor_dataset["OTI"].clip(lower=0)

sensor_dataset["target"] = (sensor_dataset["OTI"] > 0).astype(int)

In [28]:
print(sensor_dataset["OTI"].describe())
print("OTI > 0 개수:", (sensor_dataset["OTI"] > 0).sum())
print("OTI > 0 비율:", (sensor_dataset["OTI"] > 0).mean())

sensor_dataset[
    ["시리얼", "자치구명", "datetime", "visitor_count", 
     "sensor_walk_area", "m2_per_person", "OTI", "target"]
].head()

count    4033906.0
mean           0.0
std            0.0
min            0.0
25%            0.0
50%            0.0
75%            0.0
max            0.0
Name: OTI, dtype: float64
OTI > 0 개수: 0
OTI > 0 비율: 0.0


,시리얼,자치구명,datetime,visitor_count,sensor_walk_area,m2_per_person,OTI,target
0,2992,양천구,2024-12-29 23:54:00,203,65634.12,323.320788,0.0,0
1,2992,양천구,2024-12-30 00:01:00,177,65634.12,370.814237,0.0,0
2,2992,양천구,2024-12-30 00:11:00,193,65634.12,340.073161,0.0,0
3,2992,양천구,2024-12-30 00:21:00,177,65634.12,370.814237,0.0,0
4,2992,양천구,2024-12-30 00:31:00,195,65634.12,336.585231,0.0,0



안녕하세요. 열린데이터광장입니다.
원천부서에 문의 결과 전달드립니다.
1. 설치 위치 및 운영 환경, 동작 방식에 따라 개소별로 조금씩 차이가 있으며, 보편적인 센서 측정범위는 아래와 같습니다.
- 2019년 설치 센서(방문자 센서코드 4000 미만, 50대)는 전파 감지방식으로 반경 50m 내외
- 2020년 설치 센서(방문자 센서코드 4000 이상, 84대)는 영상 검지방식으로 센서 아래 정사각형 면적(2.5m x 2.5m)입니다.
2. 센서 동작방식에 따라 측정범위가 다르므로, 동일 시간에 각 지역의 절댓값을 비교하는 것이 아닌 동일 지점에서 시간 변화에 따른 추이 확인용으로 사용 부탁드립니다.
감사합니다.


> S-DoT 유동인구 센서는 설치 시기와 동작 방식에 따라 측정 범위가 다르다.\
방문자 센서코드 4000 미만의 2019년 설치 센서는 전파 감지 방식으로 반경 약 50m 내외를 측정하며,\
방문자 센서코드 4000 이상의 2020년 설치 센서는 영상 검지 방식으로 센서 하부의 정사각형 영역(2.5m × 2.5m)을 측정한다.\
본 연구에서는 이를 반영하여 센서별 유효 관측 면적을 다르게 정의하였다.\
또한 보도 폭은 임의값을 사용하지 않고, 보도 세부 현황 데이터의 면적과 연장 정보를 이용해 자치구별 평균 보도 폭을 산출하였다.

In [29]:
space2_clean = space2.copy()
space2_clean.columns = space2_clean.columns.str.strip()

space2_clean["자치구코드"] = space2_clean["관리기관"].map(gu_code_map)

space2_clean["면적_m2"] = pd.to_numeric(
    space2_clean["면적(m2)"].astype(str).str.replace(",", "", regex=False).str.strip(),
    errors="coerce"
)

space2_clean["연장_m"] = pd.to_numeric(
    space2_clean["연장(m)"].astype(str).str.replace(",", "", regex=False).str.strip(),
    errors="coerce"
)

space2_clean = space2_clean.dropna(subset=["자치구코드", "면적_m2", "연장_m"])
space2_clean = space2_clean[(space2_clean["면적_m2"] > 0) & (space2_clean["연장_m"] > 0)].copy()
space2_clean["자치구코드"] = space2_clean["자치구코드"].astype(int)

gu_width = (
    space2_clean
    .groupby("자치구코드")
    .agg(
        total_sidewalk_area=("면적_m2", "sum"),
        total_sidewalk_length=("연장_m", "sum")
    )
    .reset_index()
)

gu_width["avg_sidewalk_width"] = (
    gu_width["total_sidewalk_area"] / gu_width["total_sidewalk_length"]
)

In [30]:
sensor_dataset = sensor_dataset.merge(
    gu_width[["자치구코드", "avg_sidewalk_width"]],
    on="자치구코드",
    how="left"
)

In [31]:
# 센서 코드 기준 방식 구분
sensor_dataset["sensor_type"] = np.where(
    sensor_dataset["시리얼"] >= 4000,
    "video",
    "radio"
)

In [32]:
# 공식 측정 범위 반영
radio_radius_m = 50

sensor_dataset["sensor_eff_area"] = np.where(
    sensor_dataset["sensor_type"] == "video",
    2.5 * 2.5,  # 영상 검지 방식: 센서 아래 2.5m x 2.5m
    sensor_dataset["avg_sidewalk_width"] * (2 * radio_radius_m)  # 전파 방식: 반경 50m → 약 100m 보행 구간
)

In [33]:
a_star = 1.39

sensor_dataset["m2_per_person"] = (
    sensor_dataset["sensor_eff_area"] / sensor_dataset["visitor_count"]
)

sensor_dataset.loc[sensor_dataset["visitor_count"] <= 0, "m2_per_person"] = np.inf

sensor_dataset["OTI"] = 1 - (sensor_dataset["m2_per_person"] / a_star)
sensor_dataset["OTI"] = sensor_dataset["OTI"].clip(lower=0)

sensor_dataset["target"] = (sensor_dataset["OTI"] > 0).astype(int)

In [34]:
print(sensor_dataset["OTI"].describe())
print("OTI > 0 개수:", (sensor_dataset["OTI"] > 0).sum())
print("OTI > 0 비율:", (sensor_dataset["OTI"] > 0).mean())

sensor_dataset[
    ["시리얼", "sensor_type", "자치구명", "visitor_count",
     "avg_sidewalk_width", "sensor_eff_area", "m2_per_person", "OTI", "target"]
].head()

count    4.033906e+06
mean     3.144778e-01
std      3.883014e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      7.501998e-01
max      9.978629e-01
Name: OTI, dtype: float64
OTI > 0 개수: 1993504
OTI > 0 비율: 0.4941870236936607


,시리얼,sensor_type,자치구명,visitor_count,avg_sidewalk_width,sensor_eff_area,m2_per_person,OTI,target
0,2992,radio,양천구,203,3.905933,390.593262,1.924105,0.0,0
1,2992,radio,양천구,177,3.905933,390.593262,2.206742,0.0,0
2,2992,radio,양천구,193,3.905933,390.593262,2.023799,0.0,0
3,2992,radio,양천구,177,3.905933,390.593262,2.206742,0.0,0
4,2992,radio,양천구,195,3.905933,390.593262,2.003042,0.0,0


In [35]:
sensor_dataset

,시리얼,datetime,자치구코드,자치구명,위도,경도,visitor_count,hour,weekday,month,...,month_cos,gu_walk_area,sensor_count,sensor_walk_area,m2_per_person,OTI,target,avg_sidewalk_width,sensor_type,sensor_eff_area
0,2992,2024-12-29 23:54:00,11470.0,양천구,37.532463,126.833076,203,23,6,12,...,1.000000,328170.6,5.0,65634.12,1.924105,0.000000,0,3.905933,radio,390.593262
1,2992,2024-12-30 00:01:00,11470.0,양천구,37.532463,126.833076,177,0,0,12,...,1.000000,328170.6,5.0,65634.12,2.206742,0.000000,0,3.905933,radio,390.593262
2,2992,2024-12-30 00:11:00,11470.0,양천구,37.532463,126.833076,193,0,0,12,...,1.000000,328170.6,5.0,65634.12,2.023799,0.000000,0,3.905933,radio,390.593262
3,2992,2024-12-30 00:21:00,11470.0,양천구,37.532463,126.833076,177,0,0,12,...,1.000000,328170.6,5.0,65634.12,2.206742,0.000000,0,3.905933,radio,390.593262
4,2992,2024-12-30 00:31:00,11470.0,양천구,37.532463,126.833076,195,0,0,12,...,1.000000,328170.6,5.0,65634.12,2.003042,0.000000,0,3.905933,radio,390.593262
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4033901,4053,2025-11-28 13:40:00,11215.0,광진구,37.548059,127.106843,33,13,4,11,...,0.866025,213179.6,5.0,42635.92,0.189394,0.863745,1,4.174140,video,6.250000
4033902,4053,2025-11-28 13:50:00,11215.0,광진구,37.548059,127.106843,24,13,4,11,...,0.866025,213179.6,5.0,42635.92,0.260417,0.812650,1,4.174140,video,6.250000
4033903,4053,2025-11-28 14:00:00,11215.0,광진구,37.548059,127.106843,25,14,4,11,...,0.866025,213179.6,5.0,42635.92,0.250000,0.820144,1,4.174140,video,6.250000
4033904,4053,2025-11-28 14:10:00,11215.0,광진구,37.548059,127.106843,19,14,4,11,...,0.866025,213179.6,5.0,42635.92,0.328947,0.763347,1,4.174140,video,6.250000


In [36]:
sensor_dataset.columns

Index(['시리얼', 'datetime', '자치구코드', '자치구명', '위도', '경도', 'visitor_count', 'hour',
       'weekday', 'month', 'hour_sin', 'hour_cos', 'weekday_sin',
       'weekday_cos', 'month_sin', 'month_cos', 'gu_walk_area', 'sensor_count',
       'sensor_walk_area', 'm2_per_person', 'OTI', 'target',
       'avg_sidewalk_width', 'sensor_type', 'sensor_eff_area'],
      dtype='str')

In [37]:
initial_dataset = sensor_dataset.copy()
initial_dataset.drop(columns=["시리얼", "자치구명", "visitor_count", "sensor_count", "sensor_walk_area", "m2_per_person", "OTI", "avg_sidewalk_width", 'sensor_type', 'sensor_eff_area'], inplace=True)
initial_dataset.to_csv("../DATA/PROCESS/target_dataset.csv")

In [38]:
initial_dataset.head()

,datetime,자치구코드,위도,경도,hour,weekday,month,hour_sin,hour_cos,weekday_sin,weekday_cos,month_sin,month_cos,gu_walk_area,target
0,2024-12-29 23:54:00,11470.0,37.532463,126.833076,23,6,12,-0.258819,0.965926,-0.781831,0.62349,-2.449294e-16,1.0,328170.6,0
1,2024-12-30 00:01:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,0.000000,1.00000,-2.449294e-16,1.0,328170.6,0
2,2024-12-30 00:11:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,0.000000,1.00000,-2.449294e-16,1.0,328170.6,0
3,2024-12-30 00:21:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,0.000000,1.00000,-2.449294e-16,1.0,328170.6,0
4,2024-12-30 00:31:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,0.000000,1.00000,-2.449294e-16,1.0,328170.6,0


In [39]:
initial_dataset.shape

(4033906, 15)